<a href="https://colab.research.google.com/github/Aashish1855/102316104_Case_Study/blob/main/102316104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Self-Pruning Neural Network
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
# 1. Custom Prunable Linear Layer
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        # Normal weights and bias
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

        # Gate parameters (learnable)
        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features))

    def forward(self, x):
        # Convert gate scores into values between 0 and 1
        gates = torch.sigmoid(self.gate_scores)

        # Apply gates to weights (this is the pruning step)
        pruned_weights = self.weight * gates

        # Standard linear operation
        return F.linear(x, pruned_weights, self.bias)

    def get_gate_values(self):
        return torch.sigmoid(self.gate_scores)

# 2. Neural Network Model
class PrunableMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = PrunableLinear(32 * 32 * 3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x):
        # Flatten image
        x = x.view(x.size(0), -1)

        # Forward pass with ReLU activations
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

    def get_all_gates(self):
        gates = []
        for layer in [self.fc1, self.fc2, self.fc3]:
            gates.append(layer.get_gate_values().view(-1))

        return torch.cat(gates)

# 3. Sparsity Loss (L1 Regularization on Gates)
def compute_sparsity_loss(model):
    gates = model.get_all_gates()
    return torch.sum(gates)  # L1 norm

# 4. Training Function
def train_one_epoch(model, device, dataloader, optimizer, lambda_reg):
    model.train()
    running_loss = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Classification loss
        cls_loss = F.cross_entropy(outputs, labels)

        # Sparsity loss
        reg_loss = compute_sparsity_loss(model)

        # Total loss
        total_loss = cls_loss + lambda_reg * reg_loss

        # Backpropagation
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()

    return running_loss / len(dataloader)

# 5. Evaluation Function
def evaluate(model, device, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total

# 6. Sparsity Calculation
def compute_sparsity(model, threshold=1e-2):
    """
    Calculates percentage of weights effectively pruned
    """
    gates = model.get_all_gates()

    total_weights = gates.numel()
    pruned_weights = (gates < threshold).sum().item()

    return 100.0 * pruned_weights / total_weights

# 7. Main Training Script
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Using device: {device}")

    # Data preprocessing
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Load CIFAR-10 dataset
    train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

    # Try different lambda values
    lambda_values = [1e-5, 1e-4, 1e-3]

    for lambda_reg in lambda_values:
        print("\n" + "=" * 50)
        print(f"Training with lambda = {lambda_reg}")
        print("=" * 50)

        model = PrunableMLP().to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)

        # Train for a few epochs
        for epoch in range(5):
            loss = train_one_epoch(model, device, train_loader, optimizer, lambda_reg)
            print(f"Epoch {epoch + 1}: Loss = {loss:.4f}")

        # Evaluate model
        accuracy = evaluate(model, device, test_loader)
        sparsity = compute_sparsity(model)

        print(f"Final Test Accuracy: {accuracy:.4f}")
        print(f"Model Sparsity: {sparsity:.2f}%")


if __name__ == "__main__":
    main()


Using device: cuda


100%|██████████| 170M/170M [00:03<00:00, 47.5MB/s]



Training with lambda = 1e-05
Epoch 1: Loss = 9.6029
Epoch 2: Loss = 8.1776
Epoch 3: Loss = 7.0710
Epoch 4: Loss = 6.1855
Epoch 5: Loss = 5.4850
Final Test Accuracy: 0.5443
Model Sparsity: 0.04%

Training with lambda = 0.0001
Epoch 1: Loss = 80.2097
Epoch 2: Loss = 66.6846
Epoch 3: Loss = 54.4034
Epoch 4: Loss = 43.7834
Epoch 5: Loss = 35.0676
Final Test Accuracy: 0.5366
Model Sparsity: 0.07%

Training with lambda = 0.001
Epoch 1: Loss = 786.4273
Epoch 2: Loss = 652.5311
Epoch 3: Loss = 529.1707
Epoch 4: Loss = 421.6284
Epoch 5: Loss = 332.6460
Final Test Accuracy: 0.5343
Model Sparsity: 0.07%
